# Rule规则引擎使用指南

本 Notebook 演示 Rule 类及相关方法的使用，包括：
- 创建和解析规则
- 规则匹配与过滤
- 规则评估与报告
- 规则集分析
- 规则组合与逻辑运算

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from hscredit.core.rules import Rule, Ruleset
from hscredit.report.ruleset_report import ruleset_report

## 1. 创建示例数据

In [ ]:
# 创建示例数据
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'score': np.random.randint(300, 800, n_samples),
    'age': np.random.randint(18, 65, n_samples),
    'income': np.random.randint(3000, 50000, n_samples),
    'target': np.random.binomial(1, 0.1, n_samples),
    'loan_amount': np.random.randint(10000, 100000, n_samples),
})

print(f"数据集样本数: {len(data)}")
print(f"坏样本率: {data['target'].mean():.2%}")
data.head()

## 2. Rule基础用法

### 2.1 创建规则

In [ ]:
# 从字符串创建规则
rule1 = Rule('score < 500')
print(f"规则1: {rule1}")

# 带描述创建规则
rule2 = Rule('age < 25', desc='年轻人规则')
print(f"规则2: {rule2}, 描述: {rule2.desc}")

# 复杂规则
rule3 = Rule('score < 500 and income < 10000')
print(f"规则3: {rule3}")

### 2.2 规则匹配与过滤

In [ ]:
# 筛选命中规则的数据
print(f"\n=== 规则1: score < 500 ===")
matched = rule1.filter(data)
print(f"命中样本数: {len(matched)} / {len(data)} ({len(matched)/len(data):.1%})")
print(f"命中样本坏样本率: {matched['target'].mean():.2%}")

# 计算命中率
hit_rate = rule1.hit_rate(data)
print(f"命中率: {hit_rate:.2%}")

### 2.3 规则报告生成

In [ ]:
# 生成规则报告（样本数口径）
print("\n=== 规则报告 - 样本数口径 ===")
report = rule1.report(datasets=data, target='target', desc='低分规则')
print(report[['分箱', '样本总数', '好样本数', '坏样本数', '坏样本率', 'LIFT值']])

In [ ]:
# 生成规则报告（金额口径）
print("\n=== 规则报告 - 金额口径 ===")
report_amount = rule1.report(
    datasets=data, 
    target='target', 
    desc='低分规则',
    amount='loan_amount'
)
print(report_amount[['分箱', '样本总数', '好样本数', '坏样本数', '坏样本率', 'LIFT值']])

## 3. 多规则分析（RuleSet）

### 3.1 创建规则集

In [ ]:
# 创建多个规则
rules = [
    Rule('score < 400', desc='极低分'),
    Rule('score < 500', desc='低分'),
    Rule('age < 25', desc='年轻人'),
    Rule('income < 8000', desc='低收入'),
    Rule('score < 500 and income < 10000', desc='低分且低收入')
]

print(f"共创建 {len(rules)} 条规则")
for i, r in enumerate(rules, 1):
    print(f"  {i}. {r.desc}: {r}")

### 3.2 规则集报告

In [ ]:
# 生成规则集报告（样本数口径）
print("=== 规则集报告 - 样本数口径 ===")
ruleset_report_df = ruleset_report(
    datasets=data,
    rules=rules,
    target='target'
)

# 显示关键列
cols = ['指标含义', '分箱', '样本总数', '样本占比', '坏样本数', '坏样本率', 'LIFT值']
print(ruleset_report_df[cols].to_string())

In [ ]:
# 生成规则集报告（金额口径）
print("\n=== 规则集报告 - 金额口径 ===")
ruleset_report_amount = ruleset_report(
    datasets=data,
    rules=rules,
    target='target',
    amount='loan_amount'
)

print(ruleset_report_amount[cols].to_string())

## 4. 规则组合与逻辑运算

### 4.1 AND组合

In [ ]:
# 创建两个规则
rule_score = Rule('score < 500')
rule_age = Rule('age < 30')

# AND组合
rule_and = rule_score & rule_age
print(f"AND组合规则: {rule_and}")

# 评估
report = rule_and.report(datasets=data, target='target', desc='低分且年轻')
print(report[['分箱', '样本总数', '坏样本率', 'LIFT值']])

### 4.2 OR组合

In [ ]:
# OR组合
rule_or = rule_score | rule_age
print(f"OR组合规则: {rule_or}")

# 评估
report = rule_or.report(datasets=data, target='target', desc='低分或年轻')
print(report[['分箱', '样本总数', '坏样本率', 'LIFT值']])

### 4.3 NOT取反

In [ ]:
# NOT取反
rule_not = ~rule_score
print(f"NOT规则: {rule_not}")

# 评估
report = rule_not.report(datasets=data, target='target', desc='非低分')
print(report[['分箱', '样本总数', '坏样本率', 'LIFT值']])

## 5. 规则评估指标说明

In [ ]:
# 查看完整的评估指标
rule = Rule('score < 450', desc='高风险规则')
report = rule.report(datasets=data, target='target')

print("=== 完整评估指标 ===")
print(report.to_string())

## 6. 总结

**Rule类核心方法**：
- Rule(expression): 从字符串创建规则
- filter(data): 筛选命中规则的数据
- hit_rate(data): 计算命中率
- report(): 生成规则评估报告

**规则组合操作**：
- &: AND逻辑组合
- |: OR逻辑组合
- ~: NOT取反

**规则集报告**：
- ruleset_report(): 批量评估多个规则
- 支持样本数口径和金额口径